In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import shap
import statsmodels.api as sm
import warnings
import json
import urllib.request
import os
warnings.filterwarnings('ignore')

# Matplotlib styling for academic paper
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})


In [ ]:

# Download Indonesia GeoJSON if not exists
geojson_path = '../data/external/indonesia-prov.geojson'
os.makedirs('../data/external', exist_ok=True)
if not os.path.exists(geojson_path):
    url = "https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia-province-simple.json"
    try:
        urllib.request.urlretrieve(url, geojson_path)
        print("GeoJSON downloaded.")
    except Exception as e:
        print("Failed to download GeoJSON:", e)


In [ ]:

import os
# Load dataset with fallback paths (works whether run from /src or project root)
paths_to_try = ['../data/processed/master_dataset_v2.parquet', 'data/processed/master_dataset_v2.parquet']
df = None
for path in paths_to_try:
    if os.path.exists(path):
        df = pd.read_parquet(path)
        break

if df is None:
    raise FileNotFoundError("master_dataset_v2.parquet not found in expected locations.")

df['tanggal'] = pd.to_datetime(df['tanggal'])


In [ ]:

# ==========================================
# FIGURE 1: Interpolation Comparison (Cabai Merah)
# ==========================================
# For demonstration, we simulate 'Before' by dropping some values from the actual data for one province
cabai = df[df['provinsi'].str.contains('JAKARTA', na=False)][['tanggal', 'harga_cabai_merah_mean']].dropna().sort_values('tanggal').reset_index(drop=True).copy()
cabai['harga_raw'] = cabai['harga_cabai_merah_mean']

# Create artificial missing values to show interpolation visually
np.random.seed(42)
drop_indices = np.random.choice(cabai.index, size=int(len(cabai)*0.3), replace=False)
cabai.loc[drop_indices, 'harga_raw'] = np.nan

# Interpolate
cabai['harga_interpolated'] = cabai['harga_raw'].interpolate(method='spline', order=3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Before
sns.histplot(cabai['harga_raw'].dropna(), kde=True, color='#e74c3c', ax=axes[0], stat='density')
axes[0].set_title("Distribusi Sebelum Interpolasi (Raw)")
axes[0].set_xlabel("Harga Cabai Merah")

# Right: After
sns.histplot(cabai['harga_interpolated'], kde=True, color='#2ecc71', ax=axes[1], stat='density')
axes[1].set_title("Distribusi Sesudah Interpolasi Spline")
axes[1].set_xlabel("Harga Cabai Merah")

plt.suptitle("Gambar 1. Perbandingan Visual Distribusi Harga (Spline Interpolation)", weight='bold', y=1.05)
plt.show()


In [ ]:

# ==========================================
# FIGURE 2: Pearson Correlation Heatmap
# ==========================================
features = ['curah_hujan_mm', 'temp_avg_c', 'kemiskinan_pct', 'korban_mengungsi',
            'harga_beras_mean', 'harga_cabai_rawit_mean', 'harga_bawang_merah_mean']
corr_df = df[features].corr()

# Rename for clean plot
corr_df.columns = ['Curah Hujan', 'Suhu Rata-rata', 'Kemiskinan', 'Korban Bencana', 'Beras', 'Cabai Rawit', 'Bawang Merah']
corr_df.index = corr_df.columns

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1, square=True)
plt.title("Gambar 2. Heatmap Korelasi Pearson", weight='bold', pad=20)
plt.show()


In [ ]:

# ==========================================
# FIGURE 3: Cross Correlation Function
# ==========================================
import statsmodels.api as sm

# Aggregate to national monthly level for CCF
nat_df = df.groupby('tanggal').agg({
    'curah_hujan_mm': 'mean',
    'harga_cabai_rawit_mean': 'mean'
}).sort_index()

# Differencing for stationarity (as mentioned in paper)
x = nat_df['curah_hujan_mm'].diff().dropna()
y = nat_df['harga_cabai_rawit_mean'].diff().dropna()

ccf_vals = sm.tsa.stattools.ccf(x, y, adjusted=False)[:13] # Lags 0-12
lags = np.arange(0, 13)

# Confidence interval threshold (~ +/- 1.96 / sqrt(N))
ci = 1.96 / np.sqrt(len(x))

plt.figure(figsize=(10, 5))
markerline, stemlines, baseline = plt.stem(lags, ccf_vals, basefmt="k-")
plt.setp(markerline, color='#34495e', markersize=8)
plt.setp(stemlines, color='#34495e', linewidth=2)

# Highlight Lag 6
plt.plot(6, ccf_vals[6], marker='o', color='#e74c3c', markersize=10, zorder=3)
plt.vlines(x=6, ymin=0, ymax=ccf_vals[6], color='#e74c3c', linewidth=3, zorder=2)
plt.annotate('Lag 6 (Efek Semai-Panen)', xy=(6, ccf_vals[6]), xytext=(6, ccf_vals[6]-0.15),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6),
             ha='center', weight='bold')

plt.axhline(ci, color='gray', linestyle='--', label='95% Confidence Interval')
plt.axhline(-ci, color='gray', linestyle='--')
plt.xlabel("Lags (Bulan)")
plt.ylabel("Korelasi Silang (r)")
plt.title("Gambar 3. Cross-Correlation Function (Curah Hujan vs Cabai Rawit)", weight='bold')
plt.xticks(lags)
plt.legend()
plt.show()


In [ ]:

# ==========================================
# FIGURE 4: FSVI PCA & Choropleth Map (2 Panels)
# ==========================================

# 1. Calculate FSVI (Food Security Vulnerability Index) at Province Level
prov_df = df.groupby('provinsi').agg({
    'kemiskinan_pct': 'mean',
    'korban_mengungsi': 'mean'
}).dropna()

scaler = StandardScaler()
scaled_data = scaler.fit_transform(prov_df)

pca = PCA(n_components=2)
pca.fit(scaled_data)

# PC1 as FSVI Score
prov_df['FSVI'] = pca.transform(scaled_data)[:, 0]

# Standardize names for mapping
prov_df.index = prov_df.index.str.upper().str.strip()

fig = plt.figure(figsize=(16, 6))

# (a) Scree Plot
ax1 = plt.subplot(1, 2, 1)
ax1.bar(['PC1', 'PC2'], pca.explained_variance_ratio_ * 100, color=['#3498db', '#bdc3c7'])
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('(a) PCA Scree Plot', weight='bold')
ax1.text(0, pca.explained_variance_ratio_[0]*100 + 2, f"{pca.explained_variance_ratio_[0]*100:.1f}%", ha='center', weight='bold')

# (b) Choropleth Map
ax2 = plt.subplot(1, 2, 2)
geojson_path = '../data/external/indonesia-prov.geojson'
if os.path.exists(geojson_path):
    gdf = gpd.read_file(geojson_path)
    gdf['Propinsi'] = gdf['Propinsi'].str.upper().str.strip()
    
    # Merge
    merged = gdf.merge(prov_df, left_on='Propinsi', right_index=True, how='left')
    
    merged.plot(column='FSVI', cmap='OrRd', linewidth=0.8, ax=ax2, edgecolor='0.8', 
                legend=True, missing_kwds={'color': 'lightgrey', 'label': 'No Data'})
    ax2.set_title('(b) Peta Food Security Vulnerability Index (FSVI)', weight='bold')
    ax2.axis('off')
else:
    ax2.text(0.5, 0.5, 'GeoJSON file not found. Map cannot be rendered.', 
             ha='center', va='center', transform=ax2.transAxes)
    ax2.axis('off')

plt.suptitle("Gambar 4. Analisis Komponen Utama (PCA) dan Indeks Kerentanan", weight='bold', y=1.05)
plt.tight_layout()
plt.show()


In [ ]:

# ==========================================
# FIGURE 5: Random Forest ML Diagnostics (4 Panels)
# ==========================================

# Prepare data
ml_df = df.dropna(subset=['harga_beras_mean', 'curah_hujan_mm', 'temp_avg_c', 
                          'kemiskinan_pct', 'kepadatan_penduduk', 'korban_mengungsi']).copy()

X = ml_df[['kemiskinan_pct', 'kepadatan_penduduk', 'curah_hujan_mm', 'korban_mengungsi', 'temp_avg_c']]
y = ml_df['harga_beras_mean']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, max_depth=15, max_features='sqrt', random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
residuals = y_test - y_pred

fig = plt.figure(figsize=(16, 12))

# (a) Feature Importance
ax1 = plt.subplot(2, 2, 1)
feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=True)
# Rename for plot
name_map = {'kemiskinan_pct': 'Persentase Kemiskinan', 'kepadatan_penduduk': 'Kepadatan Penduduk',
            'curah_hujan_mm': 'Curah Hujan', 'korban_mengungsi': 'Korban Bencana', 'temp_avg_c': 'Suhu Rata-rata'}
feat_imp['Feature'] = feat_imp['Feature'].map(name_map)
ax1.barh(feat_imp['Feature'], feat_imp['Importance'], color='#2ecc71')
ax1.set_title('(a) Feature Importance', weight='bold')

# (b) Predicted vs Actual
ax2 = plt.subplot(2, 2, 2)
ax2.scatter(y_test, y_pred, alpha=0.5, color='#3498db')
# 45 degree line
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax2.plot(limits, limits, 'r--', lw=2)
ax2.set_xlabel('Actual Harga Beras')
ax2.set_ylabel('Predicted Harga Beras')
ax2.set_title('(b) Predicted vs Actual', weight='bold')

# (c) Residual Distribution
ax3 = plt.subplot(2, 2, 3)
sns.histplot(residuals, kde=True, ax=ax3, color='#9b59b6')
ax3.axvline(0, color='r', linestyle='--', lw=2)
ax3.set_xlabel('Residuals (Actual - Predicted)')
ax3.set_title('(c) Residual Distribution', weight='bold')

# (d) SHAP Summary (Beeswarm)
ax4 = plt.subplot(2, 2, 4)
# SHAP takes over the plot slightly, so we tell it not to show immediately
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
# Rename columns in X_test for SHAP plot display
X_test_renamed = X_test.rename(columns=name_map)
shap.summary_plot(shap_values, X_test_renamed, show=False, color_bar=True, max_display=5)
plt.title('(d) SHAP Summary (Beeswarm)', weight='bold')

plt.tight_layout()
plt.suptitle("Gambar 5. Diagnostik Model Random Forest", weight='bold', y=1.02)
plt.show()



# 📊 Auto-Generate Markdown Tables
Run the cell below to print the generated CSV tables as Markdown. You can easily copy and paste them directly into your paper!


In [ ]:

from IPython.display import display, Markdown

tables_to_load = [
    ("Tabel 1. Data Quality & Volatilitas", "../outputs/tables/Table_1_Data_Quality.csv"),
    ("Tabel 2. Signifikansi CCF Lags", "../outputs/tables/Table_2_CCF_Significance.csv"),
    ("Tabel 3. PCA Loadings (FSVI)", "../outputs/tables/Table_3_PCA_Loadings.csv"),
    ("Tabel 4. Performa Model (R2 & Error)", "../outputs/tables/Table_4_Model_Comparison.csv")
]

for title, path in tables_to_load:
    if os.path.exists(path):
        df_table = pd.read_csv(path)
        display(Markdown(f"### {title}"))
        # Using to_markdown() requires 'tabulate' library, if not available we use a simple fallback
        try:
            display(Markdown(df_table.to_markdown(index=False)))
        except ImportError:
            # Fallback if tabulate is not installed
            header = "| " + " | ".join(df_table.columns) + " |"
            separator = "| " + " | ".join(["---"] * len(df_table.columns)) + " |"
            rows = ["| " + " | ".join(map(str, row)) + " |" for row in df_table.values]
            display(Markdown("\n".join([header, separator] + rows)))
        display(Markdown("<br>"))
